# Evans Index X (EI-x) — MRI Adaptation

Measures the ratio of the maximum width of the frontal horns of the lateral ventricles to the maximum inner diameter of the skull, on an axial slice at or above the Anterior Commissure (AC).

**MRI adaptation**: No Hounsfield Unit (HU) windowing — Otsu thresholding operates directly on raw T1w MRI intensities. Brain tissue is bright, CSF/ventricles are dark in T1w.

**Reference**: [Samadani et al., JNS 2024](https://thejns.org/view/journals/j-neurosurg/141/3/article-p822.xml)

## Library imports

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import cv2 as cv
import SimpleITK as sitk
from skimage import measure, filters
import matplotlib.pyplot as plt

from mri_utils import (rotate_image, translate_image, ellipse_mask, getLargestCC,
                       OASIS_AC_VOXEL, OASIS_PC_VOXEL, parse_acpc_coords, load_acpc_from_txt,
                       normalize_to_uint8)

## Configuration

In [ ]:
# ── Choose your dataset ──────────────────────────────────────────────
DATASET = "oasis1"  # "oasis1", "abcd", or "custom"

# ── Dataset-specific settings ────────────────────────────────────────
if DATASET == "oasis1":
    DATA_DIR = "data/oasis1/disc1"
    DEMOGRAPHICS_PATH = "data/oasis1/oasis_cross-sectional-5708aa0a98d82080.xlsx"
    SCAN_GLOB = "OAS1_*/PROCESSED/MPRAGE/T88_111/*_masked_gfc.img"
    SUBJECT_ID_FROM_PATH = lambda p: p.split(os.sep)[-5]

elif DATASET == "abcd":
    DATA_DIR = "data/abcd"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*_T1w_acpc.nii*"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).split("_ses")[0]

elif DATASET == "custom":
    DATA_DIR = "data/my_dataset"
    DEMOGRAPHICS_PATH = None
    SCAN_GLOB = "**/*.nii.gz"
    SUBJECT_ID_FROM_PATH = lambda p: os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")

else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}")

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Dataset: {DATASET} | Data dir: {DATA_DIR}")

## Processing parameters

In [ ]:
ei_calc_processing_params = {
    "ei_x_num_slices_above_ac_to_eval": 20,
    "around_lv_roi_col_left_prcntl_lim": 20,
    "around_lv_roi_col_right_prcntl_lim": 80,
    "around_lv_roi_row_upper_prcntl_lim": 5,
    "around_lv_roi_row_lower_prcntl_lim": 95,
    "around_lv_roi_disk_size": 10,
    "axial_adaptive_thresh_size": 105,
    "axial_adaptive_thresh_mean_thresh": 7.5,
    "otsu_adaptive_diff_thresh": 0.35,
    "otsu_adaptive_horizontal_thresh": 7,
    "lv_h_roi_col_thresh_prcnt": 0.075,
    "lv_h_roi_top_row_lim": 40,
    "lv_h_roi_bot_row_lim": 15,
}

## Core function: `ei_calc()`

In [ ]:
def ei_calc(axial_data, ac_indices, ei_calc_processing_params, visualize=False):
    """Calculates Evans Index-x from axial MRI data.
    
    Iterates through axial slices above AC to find maximal frontal horn width,
    normalized by inner skull diameter at the same slice.
    
    Returns (ei_x, plane_opt) or (None, None) if extraction fails.
    """
    ei_x_num_slices_above_ac_to_eval = ei_calc_processing_params["ei_x_num_slices_above_ac_to_eval"]
    around_lv_roi_col_left_prcntl_lim = ei_calc_processing_params["around_lv_roi_col_left_prcntl_lim"]
    around_lv_roi_col_right_prcntl_lim = ei_calc_processing_params["around_lv_roi_col_right_prcntl_lim"]
    around_lv_roi_row_upper_prcntl_lim = ei_calc_processing_params["around_lv_roi_row_upper_prcntl_lim"]
    around_lv_roi_row_lower_prcntl_lim = ei_calc_processing_params["around_lv_roi_row_lower_prcntl_lim"]
    around_lv_roi_disk_size = ei_calc_processing_params["around_lv_roi_disk_size"]
    axial_adaptive_thresh_size = ei_calc_processing_params["axial_adaptive_thresh_size"]
    axial_adaptive_thresh_mean_thresh = ei_calc_processing_params["axial_adaptive_thresh_mean_thresh"]
    otsu_adaptive_diff_thresh = ei_calc_processing_params["otsu_adaptive_diff_thresh"]
    otsu_adaptive_horizontal_thresh = ei_calc_processing_params["otsu_adaptive_horizontal_thresh"]
    lv_h_roi_col_thresh_prcnt = ei_calc_processing_params["lv_h_roi_col_thresh_prcnt"]
    lv_h_roi_top_row_lim = ei_calc_processing_params["lv_h_roi_top_row_lim"]
    lv_h_roi_bot_row_lim = ei_calc_processing_params["lv_h_roi_bot_row_lim"]

    start_slice = ac_indices[2]
    max_fh_width = 0
    fh_width_list = []
    axial_brain_im_opt = None
    full_brain_mask_opt = None
    plane_opt = None
    lv_x_opt, lv_y_opt = 0, 0
    rv_x_opt, rv_y_opt = 0, 0

    for plane in range(start_slice, start_slice + ei_x_num_slices_above_ac_to_eval):
        if plane >= axial_data.shape[0]:
            break

        axial_brain_im = axial_data[plane, :, :]
        axial_uint8 = normalize_to_uint8(axial_brain_im)
        thresh = filters.threshold_otsu(axial_uint8)
        axial_bin = np.where(axial_uint8 > thresh, 1, 0)

        M0 = cv.moments(np.uint8(axial_bin))
        if M0["m00"] == 0:
            fh_width_list.append(0)
            continue

        cX = int(M0["m10"] / M0["m00"])
        cY = int(M0["m01"] / M0["m00"])
        [a_r, a_c] = axial_bin.nonzero()

        if len(a_c) == 0 or len(a_r) == 0:
            fh_width_list.append(0)
            continue

        otsu_bin_col_range = np.percentile(a_c, around_lv_roi_col_right_prcntl_lim) - np.percentile(a_c, around_lv_roi_col_left_prcntl_lim)
        otsu_bin_row_range = np.percentile(a_r, around_lv_roi_row_lower_prcntl_lim) - np.percentile(a_r, around_lv_roi_row_upper_prcntl_lim)

        a = otsu_bin_col_range // 2
        b = otsu_bin_row_range // 2
        if otsu_bin_row_range > otsu_bin_col_range:
            a, b = b, a

        around_ventricle_mask = (ellipse_mask((cY, cX), a, b, axial_bin.shape) -
                                 ellipse_mask((cY, cX), a - around_lv_roi_disk_size, b - around_lv_roi_disk_size, axial_bin.shape))

        mask_for_full_brain = np.logical_or(axial_bin, around_ventricle_mask)

        im_th = mask_for_full_brain.copy()
        h, w = im_th.shape[:2]
        mask = np.zeros((h + 2, w + 2), np.uint8)
        im_floodfill = im_th.copy()
        res = cv.floodFill(np.uint8(im_floodfill), mask, (0, 0), 255)
        full_brain_mask = (1 - res[2]).copy()
        full_brain_mask = full_brain_mask[1:h + 1, 1:w + 1]
        M = cv.moments(np.uint8(full_brain_mask))
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])

        axial_ad_th = cv.adaptiveThreshold(axial_uint8, 1, cv.ADAPTIVE_THRESH_MEAN_C,
                                           cv.THRESH_BINARY, axial_adaptive_thresh_size, axial_adaptive_thresh_mean_thresh)

        background_pixels = (1 - full_brain_mask)
        axial_brain_mask = np.where(background_pixels > 0, 0, axial_ad_th)

        adaptive_ventricles = np.where(full_brain_mask - axial_brain_mask > 0, 1, 0)
        tm = np.zeros(adaptive_ventricles.shape)
        tm[:adaptive_ventricles.shape[0] // 2, :] = 1

        diff_otsu_adaptive = np.zeros_like(adaptive_ventricles)

        try:
            otsu_ventricles = getLargestCC(measure.label(np.where(full_brain_mask - axial_bin > 0, 1, 0), background=0))
            diff_otsu_adaptive = np.where(otsu_ventricles - adaptive_ventricles > 0, 1, 0) * tm

            if (np.sum(diff_otsu_adaptive) > 0) and (np.sum((adaptive_ventricles) * tm) > 0):
                Ma = cv.moments(np.uint8(adaptive_ventricles) * tm)
                aX = int(Ma["m10"] / Ma["m00"]) if Ma["m00"] != 0 else 0
                aY = int(Ma["m01"] / Ma["m00"]) if Ma["m00"] != 0 else 0

                diff_density = np.sum(diff_otsu_adaptive) / np.sum(otsu_ventricles * tm)
                Md = cv.moments(np.uint8(diff_otsu_adaptive))
                dX = int(Md["m10"] / Md["m00"]) if Md["m00"] != 0 else 0
                dY = int(Md["m01"] / Md["m00"]) if Md["m00"] != 0 else 0

                diff_check = (diff_density > otsu_adaptive_diff_thresh) and (np.abs(aX - dX) < otsu_adaptive_horizontal_thresh)

                if diff_check:
                    ventricles = np.logical_or(adaptive_ventricles, diff_otsu_adaptive)
                else:
                    ventricles = adaptive_ventricles
            else:
                ventricles = adaptive_ventricles
        except (ValueError, IndexError, ZeroDivisionError):
            ventricles = adaptive_ventricles

        if np.sum(ventricles) == 0:
            fh_width_list.append(0)
            continue

        h_roi = np.zeros(ventricles.shape)
        [r_v, c_v] = ventricles.nonzero()
        [fr, fc] = full_brain_mask.nonzero()
        if len(fc) == 0:
            fh_width_list.append(0)
            continue

        fbc_range = max(fc) - min(fc)
        col_thresh = int(fbc_range * lv_h_roi_col_thresh_prcnt)

        r_start = max(0, cY - lv_h_roi_top_row_lim)
        r_end = max(0, cY - lv_h_roi_bot_row_lim)
        c_start = max(0, cX - col_thresh)
        c_end = min(ventricles.shape[1], cX + col_thresh)
        h_roi[r_start:r_end, c_start:c_end] = 1

        ventricles_uh = ventricles.copy()
        ventricles_uh[cY:, :] = 0
        ventricles_connected = np.logical_or(ventricles_uh, h_roi)

        try:
            [r, c] = (1 - getLargestCC(measure.label(ventricles_connected, background=0))).nonzero()
            ventricles_fh = ventricles_uh.copy()
            ventricles_fh[r, c] = 0
        except:
            fh_width_list.append(0)
            continue

        fh_roi = np.ones(ventricles_fh.shape)
        fh_roi[ac_indices[1]:, :] = 0
        ventricles_connected_fh = fh_roi * ventricles_fh

        [v_row, v_col] = ventricles_connected_fh.nonzero()
        if len(v_row) == 0:
            fh_width_list.append(0)
            continue

        top_row = min(v_row)
        top_cols = sorted(v_col[v_row == top_row])
        max_dist = top_cols[-1] - top_cols[0]
        lv_x, lv_y = top_cols[0], top_row
        rv_x, rv_y = top_cols[-1], top_row

        for rows in np.unique(sorted(v_row)):
            top_cols = sorted(v_col[v_row == rows])
            dist = top_cols[-1] - top_cols[0]
            if dist > max_dist:
                max_dist = dist
                lv_x, lv_y = top_cols[0], rows
                rv_x, rv_y = top_cols[-1], rows

        fh_width = rv_x - lv_x
        fh_width_list.append(fh_width)
        if fh_width > max_fh_width:
            max_fh_width = fh_width
            axial_brain_im_opt = axial_brain_im
            full_brain_mask_opt = full_brain_mask
            plane_opt = plane
            lv_x_opt, lv_y_opt = lv_x, lv_y
            rv_x_opt, rv_y_opt = rv_x, rv_y

    if max_fh_width == 0 or axial_brain_im_opt is None:
        print("Warning: Could not isolate frontal horns to calculate EI-x.")
        return None, None

    row, col = axial_brain_im_opt[axial_brain_im_opt.shape[0] // 2:, :].nonzero()
    if len(col) == 0:
        return None, None

    brain_l_x = min(col)
    brain_l_y = max(row[col == brain_l_x]) + axial_brain_im_opt.shape[0] // 2
    brain_r_y = brain_l_y
    brain_r_x = max(col[row == max(row[col == brain_l_x])])

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(axial_brain_im_opt, cmap='gray')
    ax.scatter([x for x in range(lv_x_opt, rv_x_opt + 1)], [lv_y_opt] * (rv_x_opt - lv_x_opt + 1),
               marker='x', s=4, color="w")
    ax.scatter([x for x in range(brain_l_x, brain_r_x + 1)], [brain_l_y] * (brain_r_x - brain_l_x + 1),
               marker='x', s=4, color="w")
    plt.show()

    brain_width = brain_r_x - brain_l_x
    ei = (max_fh_width) / brain_width if brain_width > 0 else -100

    return ei, plane_opt

## Load AC/PC coordinates

In [ ]:
def get_ac_for_subject(scan_path, subject_id):
    """Returns AC voxel indices [i, j, k] for the given subject."""
    if DATASET == "oasis1":
        # OASIS T88 atlas: all subjects share the same AC voxel location
        return list(OASIS_AC_VOXEL)
    elif DATASET == "abcd":
        # ABCD: load from per-subject AC/PC txt file
        acpc_txt = scan_path.replace("_T1w_acpc.nii.gz", "_acpc_coords.txt").replace("_T1w_acpc.nii", "_acpc_coords.txt")
        ac, pc = load_acpc_from_txt(acpc_txt)
        # Convert physical coords to voxel indices
        img = sitk.ReadImage(scan_path)
        img_rot = rotate_image(img)
        ac_idx = img_rot.TransformPhysicalPointToIndex(ac)
        return list(ac_idx)
    else:
        raise NotImplementedError(f"AC loading not configured for DATASET={DATASET}")

## Discover scans

In [ ]:
scan_files = sorted(glob.glob(os.path.join(DATA_DIR, SCAN_GLOB), recursive=True))
print(f"Found {len(scan_files)} scans")
if scan_files:
    print(f"First: {scan_files[0]}")

## Single-subject test

In [ ]:
test_path = scan_files[0]
test_id = SUBJECT_ID_FROM_PATH(test_path)
print(f"Testing on: {test_id}")

# Load and orient
img = sitk.ReadImage(test_path)
img_rot = rotate_image(img)
# No windowing — key MRI adaptation
img_arr = sitk.GetArrayFromImage(img_rot)

# Get AC coordinates
ac_indices = get_ac_for_subject(test_path, test_id)
print(f"AC voxel indices: {ac_indices}")

# Translate so AC is at origin
img_translated = translate_image(img_rot, 3, [0, 0, 0])  # identity for OASIS (already in atlas space)
ax_data = sitk.GetArrayFromImage(img_translated)

# Show mid-axial slice
mid_slice = ax_data[ac_indices[2], :, :]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
ax1.imshow(mid_slice, cmap='gray')
ax1.set_title(f"Axial slice at AC (z={ac_indices[2]})")
ax2.hist(mid_slice[mid_slice > 0].flatten(), bins=100)
ax2.set_title("Intensity histogram")
plt.tight_layout()
plt.show()

In [ ]:
ei_x, plane_opt = ei_calc(ax_data, ac_indices, ei_calc_processing_params, visualize=True)
print(f"EI-x = {ei_x:.4f}" if ei_x is not None else "EI-x computation failed")
print(f"Optimal plane: {plane_opt}")

## Batch processing

In [ ]:
results = []

for scan_path in scan_files:
    subject_id = SUBJECT_ID_FROM_PATH(scan_path)
    
    try:
        img = sitk.ReadImage(scan_path)
        img_rot = rotate_image(img)
        ax_data = sitk.GetArrayFromImage(img_rot)
        
        ac_indices = get_ac_for_subject(scan_path, subject_id)
        
        ei_x, plane_opt = ei_calc(ax_data, ac_indices, ei_calc_processing_params, visualize=False)
        plt.close('all')
        
        results.append({"subject_id": subject_id, "ei_x": ei_x, "plane_opt": plane_opt})
        print(f"{subject_id}: EI-x = {ei_x:.4f}" if ei_x is not None else f"{subject_id}: FAILED")
        
    except Exception as e:
        print(f"{subject_id}: ERROR — {e}")
        results.append({"subject_id": subject_id, "ei_x": None, "plane_opt": None})

results_df = pd.DataFrame(results)
print(f"\nProcessed {len(results_df)} subjects, {results_df['ei_x'].notna().sum()} successful")

## Results analysis

In [ ]:
# Load demographics if available
if DEMOGRAPHICS_PATH and os.path.exists(DEMOGRAPHICS_PATH):
    demo_df = pd.read_excel(DEMOGRAPHICS_PATH) if DEMOGRAPHICS_PATH.endswith('.xlsx') else pd.read_csv(DEMOGRAPHICS_PATH)
    print(f"Demographics loaded: {len(demo_df)} rows")
else:
    demo_df = None

# Save to CSV
output_path = os.path.join(OUTPUT_DIR, f"ei_x_{DATASET}.csv")
results_df.to_csv(output_path, index=False)
print(f"Results saved to {output_path}")

# Distribution plot
fig, ax = plt.subplots(figsize=(10, 5))
valid = results_df['ei_x'].dropna()
ax.hist(valid, bins=20, edgecolor='black', alpha=0.7)
ax.set_xlabel("Evans Index X")
ax.set_ylabel("Count")
ax.set_title(f"EI-x Distribution ({DATASET}, n={len(valid)})")
ax.axvline(x=0.3, color='r', linestyle='--', label='Clinical threshold (0.30)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean: {valid.mean():.4f}, Std: {valid.std():.4f}")
print(f"Min: {valid.min():.4f}, Max: {valid.max():.4f}")